In [3]:
%run "C:/Users/justin.diener/OneDrive/LACO/Given Training/Pyspark/Wills pyspark git code/wills_pyspark_training/_local_dev/0 - Create local Spark session.ipynb"

# 03 - Filtering and Creating Columns

This lesson turns a small typed order-line DataFrame into a smaller, clearer business result.

## Learning objectives

By the end of this notebook, you will be able to:

- reference columns and fixed values with `F.col` and `F.lit`;
- filter with one or more conditions;
- calculate and classify values; and
- rename, remove, select, and sort columns.

## Prerequisite recap

Notebook 02 introduced schemas, selecting columns, aliases, and DataFrame immutability. Spark expressions operate on columns row by row without changing the original DataFrame.

In [ ]:
from pyspark.sql import functions as F

order_lines = spark.createDataFrame(
    [
        (1001, 'North', 'Stationery', 2, 18.50),
        (1002, 'West', 'Furniture', 1, 750.00),
        (1003, 'North', 'Stationery', 3, 12.00),
        (1004, 'South', 'Electronics', 2, 85.00),
        (1005, 'East', 'Furniture', 2, 420.00),
    ],
    'order_id INT, region STRING, category STRING, quantity INT, unit_price DOUBLE',
)
order_lines.show()

## Column expressions

`F.col('quantity')` means use values from that column. `F.lit('training')` supplies the same fixed value for every row. These expressions can be combined into calculations and conditions.

## Filter rows

Use `&` for **and**, `|` for **or**, and `~` for **not**. Put each condition in parentheses. `isin` is concise when several values are acceptable.

In [ ]:
focus_orders = order_lines.filter(
    F.col('region').isin('North', 'West')
    & ~(F.col('category') == 'Furniture')
)
focus_orders.show()

## Your turn

Create `non_northern_order` from `order_lines` by selecting all orders that are not from the `North` region

In [ ]:
# Write your solution here.


## Create calculated and conditional columns

`withColumn` adds or replaces a column. `when(...).otherwise(...)` is the DataFrame equivalent of SQL `CASE WHEN ... ELSE ... END`. Chained conditions are checked from top to bottom.

In [ ]:
enriched_orders = (
    order_lines
    .withColumn('order_value', F.round(F.col('quantity') * F.col('unit_price'), 2))
    .withColumn(
        'value_band',
        F.when(F.col('order_value') >= 500, 'High')
        .when(F.col('order_value') >= 100, 'Medium')
        .otherwise('Standard'),
    )
    .withColumn('source_system', F.lit('training'))
)
enriched_orders.show()

## Your turn

Create `discounted_orders` from `enriched_orders`:

- add `discount_rate`: 10% for `High`, 5% for `Medium`, and 0% for `Standard`;
- calculate `discount_amount` as `order_value * discount_rate`;
- calculate `net_order_value` as `order_value - discount_amount`; and
- select `order_id`, `region`, `value_band`, `order_value`, `discount_rate`, `discount_amount`, and `net_order_value`

In [ ]:
# Write your solution here.

### Expected result

All five orders remain. Orders 1005 and 1002 receive a 10% discount, order 1004 receives a 5% discount, and the two `Standard` orders receive no discount. The first row is order 1005 with `net_order_value` 756.00.

In [26]:
discounted_orders = (
    enriched_orders
    .withColumn(
        'discount_rate',
        F.when(F.col('value_band') == 'High', F.lit(0.10))
        .when(F.col('value_band') == 'Medium', F.lit(0.05))
        .otherwise(F.lit(0.00)),
    )
    .withColumn(
        'discount_amount',
        F.col('order_value') * F.col('discount_rate'),
    )
    .withColumn(
        'net_order_value',
        F.col('order_value') - F.col('discount_amount'),
    )
    .select(
        'order_id', 'region', 'value_band', 'order_value',
        'discount_rate', 'discount_amount', 'net_order_value',
    )
    .orderBy(F.col('net_order_value').desc())
)
discounted_orders.show()

+--------+------+----------+-----------+-------------+---------------+---------------+
|order_id|region|value_band|order_value|discount_rate|discount_amount|net_order_value|
+--------+------+----------+-----------+-------------+---------------+---------------+
|    1005|  East|      High|      840.0|          0.1|           84.0|          756.0|
|    1002|  West|      High|      750.0|          0.1|           75.0|          675.0|
|    1004| South|    Medium|      170.0|         0.05|            8.5|          161.5|
|    1001| North|  Standard|       37.0|          0.0|            0.0|           37.0|
|    1003| North|  Standard|       36.0|          0.0|            0.0|           36.0|
+--------+------+----------+-----------+-------------+---------------+---------------+



## Rename, remove, and sort

`withColumnRenamed` changes a column name in the new DataFrame. `drop` removes columns. `orderBy` sorts rows; use `desc()` for descending order.

In [ ]:
order_report = (
    enriched_orders
    .withColumnRenamed('region', 'sales_region')
    .drop('unit_price')
    .orderBy(F.col('order_value').desc())
)
order_report.show()

## Your turn

Create `high_value_orders` from `order_lines`:

- calculate `order_value`;
- keep non-South orders worth at least 500;
- label values of at least 800 as `Very high` and the rest as `High`;
- add `source_system = 'training'`;
- rename `order_id` to `sales_order_id`; and
- select useful columns and sort by order value descending.

In [ ]:
# Write your solution here.

### Expected result

Two rows remain, ordered as sales order 1005 with value 840 and `Very high`, then sales order 1002 with value 750 and `High`.

### Solution - reveal after attempting

In [ ]:
high_value_orders = (
    order_lines
    .withColumn('order_value', F.round(F.col('quantity') * F.col('unit_price'), 2))
    .filter((F.col('region') != 'South') & (F.col('order_value') >= 500))
    .withColumn(
        'value_band',
        F.when(F.col('order_value') >= 800, 'Very high').otherwise('High'),
    )
    .withColumn('source_system', F.lit('training'))
    .withColumnRenamed('order_id', 'sales_order_id')
    .select('sales_order_id', 'region', 'order_value', 'value_band', 'source_system')
    .orderBy(F.col('order_value').desc())
)
high_value_orders.show()

## Key takeaway

Build readable transformations one expression at a time, keep conditions parenthesised, and assign the result to a descriptive name.

**Next:** convert raw text into useful Spark data types.